In [1]:
#  LASSO implemented in model to predict ridership

In [2]:
## Packages

In [3]:
# Data handling
import pandas as pd
import geopandas as gpd
import numpy as np

# Geopandas Dependency
import pyarrow

# Data reading
import requests
from io import BytesIO, StringIO

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import leafmap.foliumap as leafmap
import plotly.express as px


# Model requirements
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier


# Model validation
from sklearn.model_selection import train_test_split, cross_val_score

In [4]:
## Read the Clean datsets

In [5]:
url_ts = "https://raw.githubusercontent.com/iansargent/nyc-subway-ridership-ml/main/Data/Clean/origin_ridership_time_series_CLEAN.csv"
origin = pd.read_csv(StringIO(requests.get(url_ts, verify=False).text))

In [6]:
# show datset
display(origin.head())

,month,day_of_week,hour_of_day,origin_station_complex_id,origin_station_complex_name,origin_latitude,origin_longitude,origin_point,sum_estimated_average_ridership
0,1,Friday,0,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),35.7473
1,1,Friday,1,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),11.4975
2,1,Friday,2,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),10.9980
3,1,Friday,3,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),14.7534
4,1,Friday,4,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,POINT (-73.912034 40.775036),52.0023


In [7]:
## Train test split

In [8]:
#split into X and Y datasets
num_cols = origin.shape[1]
X = origin.iloc[:,0:num_cols-2]
X = X.drop('origin_station_complex_id', axis=1) #drop complex_id b/c it is the same as complex_name in terms of analysis
X = pd.get_dummies(X, drop_first=True) #one_hot encoding here
Y = origin.iloc[:,num_cols-1:num_cols]
Y = np.ravel(Y)

In [9]:
from sklearn.preprocessing import OneHotEncoder

#split into X and Y datasets
num_cols = origin.shape[1]
X = origin.iloc[:,0:num_cols-2]
X = X.drop('origin_station_complex_id', axis=1) #drop complex_id b/c it is the same as complex_name in terms of analysis
#one hot encoding
categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
encoder = OneHotEncoder(sparse_output=False, drop = 'first')
one_hot_encoded = encoder.fit_transform(X[categorical_columns])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns))
df_encoded = pd.concat([X, one_hot_df], axis=1)
df_encoded = df_encoded.drop(categorical_columns, axis=1)
X = df_encoded

Y = origin.iloc[:,num_cols-1:num_cols]
Y = np.ravel(Y)


display(X)

,month,hour_of_day,origin_latitude,origin_longitude,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday,...,"origin_station_complex_name_West Farms Sq-E Tremont Av (2,5)",origin_station_complex_name_Westchester Sq-E Tremont Av (6),origin_station_complex_name_Whitlock Av (6),origin_station_complex_name_Wilson Av (L),"origin_station_complex_name_Winthrop St (2,5)","origin_station_complex_name_Woodhaven Blvd (J,Z)","origin_station_complex_name_Woodhaven Blvd (M,R)",origin_station_complex_name_Woodlawn (4),origin_station_complex_name_York St (F),origin_station_complex_name_Zerega Av (6)
0,1,0,40.775036,-73.912034,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1,40.775036,-73.912034,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,2,40.775036,-73.912034,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,3,40.775036,-73.912034,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,4,40.775036,-73.912034,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852928,12,19,40.692259,-73.986642,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
852929,12,20,40.692259,-73.986642,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
852930,12,21,40.692259,-73.986642,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
852931,12,22,40.692259,-73.986642,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
#train test split happening
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=4)

In [11]:
## End of Atticus's code

In [12]:
## LASSO Model

In [13]:
# standardize
from sklearn.preprocessing import StandardScaler

scalerX = StandardScaler().fit(X_train) 
X_train_scaled = scalerX.transform(X_train)
X_test_scaled = scalerX.transform(X_test)



In [ ]:
# 5-fold cross-validation to choose lambda
#from sklearn.linear_model import LassoCV

#lasso_cv_model = LassoCV(
#    alphas=np.arange(0.01, 1, 0.1), cv=5, max_iter=100000
#)

# fit model on train set
#lasso_best = lasso_cv_model.fit(X_train_scaled, y_train)

In [ ]:
# chosen value of lambda
#lasso_best.alpha_

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Starting the lasso regression engine
lasso_10 = Lasso(alpha = 10)
lasso_10.fit(X_train, Y_train)
lasso_coefs10 = lasso_10.coef_

linear_model = LinearRegression().fit(X_train, Y_train)
linear_coefs = linear_model.coef_

# Most of the differences are 0's
pd.DataFrame({
    'term' : X_train.columns,
    'OLS'  : linear_coefs,
    'Lasso': lasso_coefs10,
    'estimate_diff': (lasso_coefs10 - linear_coefs)#.round().astype(int)
})#.query('estimate_diff != 0')

In [ ]:
# how many coeffs aren't zero
sum(np.abs(lasso_best.coef_) > 0)

In [ ]:
# Getting the coefficients of the model for the best choice of lambda
pd.DataFrame({
    'feature' : X.columns,
    'estimate': lasso_best.coef_.round(2)
}).query('estimate != 0')

In [ ]:
# fit mlr model with significant predictors
best_mlr = (
    LinearRegression()
    .fit(
        X = X[[]],
        y = Y
    )
)

# Creating an R-squared plot:
sns.lmplot(
    data = pd.DataFrame({
        'y': Y,
        'y_hat': best_mlr.predict(X = X[[]])
    }),
    x = 'y_hat',
    y = 'y',
    line_kws = {'color': 'red'},
    scatter_kws = {'s' : 10}
)

plt.show()